In [7]:
   !pip install pgmpy

In [8]:
import warnings
warnings.filterwarnings("ignore")

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD

model = DiscreteBayesianNetwork([
    ("rain", "maintenance"),
    ("rain", "train"),
    ("maintenance", "train"),
    ("train", "appointment"),
])

rain = TabularCPD(
    variable="rain", variable_card=3,
    values=[[0.7], [0.2], [0.1]],
    state_names={"rain": ["none", "light", "heavy"]})

maintenance = TabularCPD(
    variable="maintenance", variable_card=2,
    values=[[0.4, 0.2, 0.1],
            [0.6, 0.8, 0.9]],
    evidence=["rain"], evidence_card=[3],
    state_names={"maintenance": ["yes", "no"],
                 "rain": ["none", "light", "heavy"]})

train = TabularCPD(
    variable="train", variable_card=2,
    values=[[0.8, 0.9, 0.6, 0.7, 0.4, 0.5],
            [0.2, 0.1, 0.4, 0.3, 0.6, 0.5]],
    evidence=["rain", "maintenance"], evidence_card=[3, 2],
    state_names={"train": ["on time", "delayed"],
                 "rain": ["none", "light", "heavy"],
                 "maintenance": ["yes", "no"]})

appointment = TabularCPD(
    variable="appointment", variable_card=2,
    values=[[0.9, 0.6],
            [0.1, 0.4]],
    evidence=["train"], evidence_card=[2],
    state_names={"appointment": ["attend", "miss"],
                 "train": ["on time", "delayed"]})

model.add_cpds(rain, maintenance, train, appointment)
print("Model is correct:", model.check_model())

Model is correct: True


In [9]:
p = (rain.get_value(rain="none")
     * maintenance.get_value(maintenance="no", rain="none")
     * train.get_value(train="on time", rain="none", maintenance="no")
     * appointment.get_value(appointment="attend", train="on time"))
print(round(p, 4))

0.3402


In [10]:
p = (rain.get_value(rain="heavy")
     * maintenance.get_value(maintenance="yes", rain="heavy")
     * train.get_value(train="delayed", rain="heavy", maintenance="yes")
     * appointment.get_value(appointment="miss", train="delayed"))
print(round(p, 4))

0.0024


In [11]:
from pgmpy.inference import VariableElimination
inference = VariableElimination(model)

def show(evidence):
    for node in ["rain", "maintenance", "train", "appointment"]:
        if node in evidence:
            print(f"{node}: {evidence[node]}   (observed)")
            continue
        result = inference.query([node], evidence=evidence, show_progress=False)
        print(node)
        for value, prob in zip(result.state_names[node], result.values):
            print(f"    {value}: {prob:.4f}")

show({"train": "delayed"})

rain
    none: 0.4601
    light: 0.3005
    heavy: 0.2394
maintenance
    yes: 0.3662
    no: 0.6338
train: delayed   (observed)
appointment
    attend: 0.6000
    miss: 0.4000


In [12]:
show({"appointment": "miss"})

rain
    none: 0.6065
    light: 0.2392
    heavy: 0.1544
maintenance
    yes: 0.3441
    no: 0.6559
train
    on time: 0.4802
    delayed: 0.5198
appointment: miss   (observed)


In [13]:
show({"train": "delayed", "rain": "heavy"})

rain: heavy   (observed)
maintenance
    yes: 0.1176
    no: 0.8824
train: delayed   (observed)
appointment
    attend: 0.6000
    miss: 0.4000


## Short answers

**Q13:** It's not luck. pgmpy uses Variable Elimination, which is exact, so it does basically the same thing I did by hand: it adds up over maintenance and then normalizes. It just does it in a smarter order. Since it's the same math, it gives the same answer, 0.4601.

**Q14:** The appointment only depends on the train. Once we already know the train is delayed, rain and maintenance don't add anything new, so we can just read the number straight from the appointment table, which is 0.4.

**Q16:** Missing the appointment obviously doesn't cause rain. But if I missed it, the train was probably delayed, and delays happen more when it rains heavily. So the evidence goes backwards up the arrows, from the appointment to the train to the rain, and that's why heavy rain goes up a bit.

**Q18:** When it's already raining heavily, the train has a high chance of being late either way: 0.6 with maintenance and 0.5 without. Those numbers are really close, so knowing the train was delayed doesn't tell us much about whether there was maintenance. That's why it only moved from 0.1 to about 0.12.